# SensorFusion-HAR: Lightweight Real-Time Human Activity Recognition

## Architecture Overview

4-stage pipeline: Echo State Network (reservoir) -> DS-Conv1D -> Patch Micro-Attention -> Binary Head

Supports UCI-HAR (6 classes) and PAMAP2 (12 complex activities). Features data augmentation, contrastive pre-training, LOSO evaluation, and comprehensive analysis.

In [ ]:
import os, json, time, copy
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.manifold import TSNE
from model import SensorFusionHAR
from model.dataset import UCIHARDataset
from model.dataset_pamap2 import PAMAP2Dataset, ACTIVITY_NAMES as PAMAP2_LABELS
from model.reservoir import EchoStateNetwork
from model.dsconv import DSConvEncoder
from model.attention import PatchMicroAttention
from model.binary_head import BinaryClassifier
from model.augmentation import SensorAugmentor, AugmentedDataset
from model.contrastive import SensorSimCLR, nt_xent_loss, pretrain_contrastive, transfer_weights
from model.visualize import plot_tsne, plot_attention_maps, plot_noise_robustness, plot_confidence_calibration

plt.style.use("dark_background")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

LABELS_6 = ["Walking", "Upstairs", "Downstairs", "Sitting", "Standing", "Laying"]
COLORS_6 = ["#4CAF50", "#FF9800", "#2196F3", "#9C27B0", "#F44336", "#00BCD4"]
COLORS_12 = ["#4CAF50", "#FF9800", "#2196F3", "#9C27B0", "#F44336", "#00BCD4", "#E91E63", "#CDDC39", "#795548", "#607D8B", "#FF5722", "#3F51B5"]

## 1. Dataset

In [ ]:
DATA_DIR = "data/UCI HAR Dataset"

if not os.path.isdir(DATA_DIR):
    UCIHARDataset.download("data")

train_ds = UCIHARDataset(DATA_DIR, split="train")
test_ds = UCIHARDataset(DATA_DIR, split="test")

print(f"UCI-HAR -- Train: {train_ds.X.shape}, Test: {test_ds.X.shape}")
for i in range(6):
    print(f"  {LABELS_6[i]}: {(train_ds.y == i).sum().item()}")

means, stds = UCIHARDataset.get_normalization_stats(DATA_DIR)
mean_t = torch.tensor(means, dtype=torch.float32)
std_t = torch.tensor(stds, dtype=torch.float32)
train_ds.X = (train_ds.X - mean_t) / std_t
test_ds.X = (test_ds.X - mean_t) / std_t

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
ch_names = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]
for idx in range(6):
    ax = axes[idx // 3][idx % 3]
    mask = train_ds.y == idx
    si = torch.where(mask)[0][0].item()
    window = train_ds.X[si].numpy()
    for ch in range(6):
        ax.plot(window[:, ch], linewidth=0.8, alpha=0.8)
    ax.set_title(LABELS_6[idx], color=COLORS_6[idx], fontsize=13, fontweight="bold")
    ax.set_xlabel("Timestep")
    if idx % 3 == 0:
        ax.set_ylabel("Value")
plt.tight_layout()
plt.savefig("sample_windows.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Data Augmentation

In [ ]:
aug = SensorAugmentor(p=1.0)
sample = train_ds.X[0].numpy().copy()

fig, axes = plt.subplots(2, 4, figsize=(20, 6))
axes[0, 0].plot(sample[:, 0], color="#4CAF50", linewidth=0.8)
axes[0, 0].set_title("Original")

aug_methods = ["jitter", "scaling", "rotation", "permutation", "time_warp", "magnitude_warp", "channel_dropout"]
for i, name in enumerate(aug_methods):
    ax = axes[(i + 1) // 4][(i + 1) % 4]
    augmented = getattr(aug, name)(sample.copy())
    ax.plot(augmented[:, 0], color="#FF9800", linewidth=0.8)
    ax.set_title(name.replace("_", " ").title())

plt.tight_layout()
plt.savefig("augmentations_demo.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Model Architecture

In [ ]:
model = SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=6).to(device)

print(model)
print()

modules = [("Reservoir (ESN)", model.reservoir), ("DS-Conv Encoder", model.dsconv), ("Patch Attention", model.attention), ("Binary Classifier", model.classifier)]
for name, mod in modules:
    params = sum(p.numel() for p in mod.parameters() if p.requires_grad)
    buffers = sum(b.numel() for b in mod.buffers())
    print(f"{name:25s} | Trainable: {params:>8,} | Buffers: {buffers:>6,}")

print(f"\nTotal trainable: {model.count_parameters():,}")
print(f"Model size (FP32): {model.model_size_kb():.2f} KB")
print(f"Model size (INT8): {model.quantized_size_kb():.2f} KB")

dummy = torch.randn(1, 128, 6).to(device)
out = model(dummy)
print(f"\nForward pass: {dummy.shape} -> {out.shape}")

## 4. Training with Data Augmentation

In [ ]:
EPOCHS = 100
LR = 0.001
BATCH_SIZE = 64

aug_train = AugmentedDataset(train_ds, SensorAugmentor(p=0.5))
train_loader = DataLoader(aug_train, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()

os.makedirs("checkpoints", exist_ok=True)
best_acc = 0.0
history = {"train_loss": [], "test_loss": [], "test_acc": [], "test_f1": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * X.size(0)
    train_loss = epoch_loss / len(aug_train)

    model.eval()
    t_loss, ap, al = 0.0, [], []
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            t_loss += criterion(out, y).item() * X.size(0)
            ap.append(out.argmax(1).cpu().numpy())
            al.append(y.cpu().numpy())
    ap, al = np.concatenate(ap), np.concatenate(al)
    test_loss = t_loss / len(test_ds)
    test_acc = (ap == al).mean()
    test_f1 = f1_score(al, ap, average="macro")
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)
    history["test_f1"].append(test_f1)

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save({"model_state_dict": model.state_dict(), "epoch": epoch, "accuracy": test_acc, "f1_macro": test_f1, "normalization_stats": {"means": means, "stds": stds}}, "checkpoints/best_model.pt")

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS} | Train: {train_loss:.4f} | Test: {test_loss:.4f} | Acc: {test_acc*100:.2f}% | F1: {test_f1:.4f}")

print(f"\nBest accuracy: {best_acc*100:.2f}%")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
ep_range = range(1, EPOCHS + 1)
best_ep = np.argmax(history["test_acc"]) + 1

axes[0, 0].plot(ep_range, history["train_loss"], color="#FF6B6B", linewidth=1.2)
axes[0, 0].set_title("Train Loss")
axes[0, 0].axvline(best_ep, color="white", linestyle="--", alpha=0.5)

axes[0, 1].plot(ep_range, history["test_loss"], color="#4ECDC4", linewidth=1.2)
axes[0, 1].set_title("Test Loss")
axes[0, 1].axvline(best_ep, color="white", linestyle="--", alpha=0.5)

axes[1, 0].plot(ep_range, [a * 100 for a in history["test_acc"]], color="#45B7D1", linewidth=1.2)
axes[1, 0].set_title("Test Accuracy (%)")
axes[1, 0].axvline(best_ep, color="white", linestyle="--", alpha=0.5)

axes[1, 1].plot(ep_range, history["test_f1"], color="#96CEB4", linewidth=1.2)
axes[1, 1].set_title("Test F1 Macro")
axes[1, 1].axvline(best_ep, color="white", linestyle="--", alpha=0.5)

for ax in axes.flat:
    ax.set_xlabel("Epoch")
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Evaluation

In [ ]:
cp = torch.load("checkpoints/best_model.pt", map_location=device, weights_only=False)
model.load_state_dict(cp["model_state_dict"])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for X, y in test_loader:
        X, y = X.to(device), y.to(device)
        out = model(X)
        all_preds.append(out.argmax(1).cpu().numpy())
        all_labels.append(y.cpu().numpy())
all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

acc = (all_preds == all_labels).mean()
f1_macro = f1_score(all_labels, all_preds, average="macro")
f1_per = f1_score(all_labels, all_preds, average=None)

print(f"Accuracy: {acc*100:.2f}%")
print(f"F1 Macro: {f1_macro:.4f}")
print(f"Best epoch: {cp['epoch']}")
print(f"\n{classification_report(all_labels, all_preds, target_names=LABELS_6)}")

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap="YlOrRd")
ax.set_xticks(range(6))
ax.set_yticks(range(6))
ax.set_xticklabels(LABELS_6, rotation=45, ha="right")
ax.set_yticklabels(LABELS_6)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix")
for i in range(6):
    for j in range(6):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color=color, fontsize=12)
plt.colorbar(im)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(range(6), f1_per, color=COLORS_6, height=0.6)
ax.set_yticks(range(6))
ax.set_yticklabels(LABELS_6)
ax.set_xlabel("F1 Score")
ax.set_title("Per-Class F1 Score")
ax.set_xlim(0, 1.05)
for bar, val in zip(bars, f1_per):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2, f"{val:.3f}", va="center", fontsize=11)
plt.tight_layout()
plt.savefig("per_class_f1.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Ablation Study

In [ ]:
class NoReservoirModel(nn.Module):
    def __init__(self, nc=6):
        super().__init__()
        self.proj = nn.Linear(6, 32)
        self.dsconv = DSConvEncoder(in_channels=32)
        self.attention = PatchMicroAttention(in_channels=48, seq_len=32, d_model=32, ff_dim=48)
        self.classifier = BinaryClassifier(in_features=32, num_classes=nc)
    def forward(self, x):
        return self.classifier(self.attention(self.dsconv(self.proj(x).transpose(1, 2))))
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

class NoAttentionModel(nn.Module):
    def __init__(self, nc=6):
        super().__init__()
        self.reservoir = EchoStateNetwork(6, 32)
        self.dsconv = DSConvEncoder(in_channels=32)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = BinaryClassifier(in_features=48, num_classes=nc)
    def forward(self, x):
        return self.classifier(self.pool(self.dsconv(self.reservoir(x).transpose(1, 2))).squeeze(-1))
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

class NoBinaryHeadModel(nn.Module):
    def __init__(self, nc=6):
        super().__init__()
        self.reservoir = EchoStateNetwork(6, 32)
        self.dsconv = DSConvEncoder(in_channels=32)
        self.attention = PatchMicroAttention(in_channels=48, seq_len=32, d_model=32, ff_dim=48)
        self.bn = nn.BatchNorm1d(32)
        self.head = nn.Linear(32, nc)
    def forward(self, x):
        x = self.attention(self.dsconv(self.reservoir(x).transpose(1, 2)))
        return self.head(self.bn(x))
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

class NoDSConvModel(nn.Module):
    def __init__(self, nc=6):
        super().__init__()
        self.reservoir = EchoStateNetwork(6, 32)
        self.conv = nn.Sequential(
            nn.Conv1d(32, 48, 5, 1, 2), nn.BatchNorm1d(48), nn.ReLU(inplace=True),
            nn.Conv1d(48, 48, 5, 2, 2), nn.BatchNorm1d(48), nn.ReLU(inplace=True),
            nn.Conv1d(48, 48, 3, 2, 1), nn.BatchNorm1d(48), nn.ReLU(inplace=True))
        self.attention = PatchMicroAttention(in_channels=48, seq_len=32, d_model=32, ff_dim=48)
        self.classifier = BinaryClassifier(in_features=32, num_classes=nc)
    def forward(self, x):
        return self.classifier(self.attention(self.conv(self.reservoir(x).transpose(1, 2))))
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

ABL_EPOCHS = 50
abl_variants = [
    ("Full Model", SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=6)),
    ("No Reservoir", NoReservoirModel()),
    ("No Attention", NoAttentionModel()),
    ("No Binary Head", NoBinaryHeadModel()),
    ("No DS-Conv", NoDSConvModel()),
]
abl_results = []
for vn, vm in abl_variants:
    vm = vm.to(device)
    vp_count = vm.count_parameters()
    print(f"\n--- {vn} ({vp_count:,} params) ---")
    vo = torch.optim.AdamW(vm.parameters(), lr=LR, weight_decay=1e-4)
    vs = torch.optim.lr_scheduler.CosineAnnealingLR(vo, T_max=ABL_EPOCHS)
    best_va = 0.0
    for ep in range(1, ABL_EPOCHS + 1):
        vm.train()
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            vo.zero_grad()
            nn.CrossEntropyLoss()(vm(X), y).backward()
            vo.step()
        vs.step()
        vm.eval()
        vpp, vll = [], []
        with torch.no_grad():
            for X, y in test_loader:
                X, y = X.to(device), y.to(device)
                vpp.append(vm(X).argmax(1).cpu().numpy())
                vll.append(y.cpu().numpy())
        vpp, vll = np.concatenate(vpp), np.concatenate(vll)
        va = (vpp == vll).mean()
        if va > best_va:
            best_va = va
        if ep % 10 == 0:
            print(f"  Epoch {ep}/{ABL_EPOCHS}: acc={va*100:.2f}%")
    vf = f1_score(vll, vpp, average="macro")
    abl_results.append((vn, vp_count, best_va, vf))
    print(f"  Best: {best_va*100:.2f}% | F1: {vf:.4f}")

print(f"\n{'='*60}")
print(f"{'Variant':<20s}{'Params':>10s}{'Accuracy':>12s}{'F1 Macro':>12s}")
print(f"{'-'*54}")
for vn, vp, va, vf in abl_results:
    print(f"{vn:<20s}{vp:>10,}{va*100:>11.2f}%{vf:>12.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
names = [r[0] for r in abl_results]
accs = [r[2] * 100 for r in abl_results]
f1s = [r[3] * 100 for r in abl_results]
y_pos = np.arange(len(names))
bh = 0.35
b1 = ax.barh(y_pos - bh / 2, accs, bh, label="Accuracy", color="#45B7D1", alpha=0.9)
b2 = ax.barh(y_pos + bh / 2, f1s, bh, label="F1 Macro", color="#96CEB4", alpha=0.9)
for bar in b1:
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2, f"{bar.get_width():.1f}%", va="center", fontsize=9)
for bar in b2:
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2, f"{bar.get_width():.1f}%", va="center", fontsize=9)
ax.set_yticks(y_pos)
ax.set_yticklabels(names)
ax.set_xlabel("Score (%)")
ax.set_title("Ablation Study Results")
ax.legend(loc="lower right")
ax.set_xlim(0, 105)
plt.tight_layout()
plt.savefig("ablation_results.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. t-SNE Feature Visualization

In [ ]:
fig = plot_tsne(model, test_ds, device, stage="all", n_samples=1000, activity_labels=LABELS_6, colors=COLORS_6, save_path="tsne_stages.png")
plt.show()

## 8. Attention Map Visualization

In [ ]:
fig = plot_attention_maps(model, test_ds, device, n_samples=4, activity_labels=LABELS_6, colors=COLORS_6, save_path="attention_maps.png")
plt.show()

## 9. Noise Robustness Analysis

In [ ]:
results, fig = plot_noise_robustness(model, test_ds, device, activity_labels=LABELS_6, save_path="noise_robustness.png")
plt.show()
print("\nNoise Robustness:")
for snr, a, f in zip(results["snr_levels"], results["accuracy"], results["f1"]):
    print(f"  SNR {snr:3d} dB: Acc={a*100:.2f}%, F1={f:.4f}")

## 10. Confidence Calibration

In [ ]:
ece, fig = plot_confidence_calibration(model, test_ds, device, activity_labels=LABELS_6, save_path="calibration.png")
plt.show()
print(f"Expected Calibration Error (ECE): {ece:.4f}")

## 11. Contrastive Pre-Training (SimCLR)

In [ ]:
print("Phase 1: Contrastive pre-training (50 epochs)...")
simclr = SensorSimCLR(input_channels=6, reservoir_size=32).to(device)
simclr_aug = SensorAugmentor(p=0.8)
simclr = pretrain_contrastive(simclr, train_ds, simclr_aug, device, epochs=50, batch_size=128, lr=0.0003)

print("\nPhase 2: Transfer + Fine-tuning (50 epochs)...")
model_ft = SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=6).to(device)
transfer_weights(simclr, model_ft)

ft_opt = torch.optim.AdamW(model_ft.parameters(), lr=0.0005, weight_decay=1e-4)
ft_sched = torch.optim.lr_scheduler.CosineAnnealingLR(ft_opt, T_max=50)
ft_best = 0.0

for ep in range(1, 51):
    model_ft.train()
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        ft_opt.zero_grad()
        nn.CrossEntropyLoss()(model_ft(X), y).backward()
        ft_opt.step()
    ft_sched.step()
    model_ft.eval()
    fp, fl = [], []
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            fp.append(model_ft(X).argmax(1).cpu().numpy())
            fl.append(y.cpu().numpy())
    fp, fl = np.concatenate(fp), np.concatenate(fl)
    ft_acc = (fp == fl).mean()
    if ft_acc > ft_best:
        ft_best = ft_acc
    if ep % 10 == 0:
        ft_f1 = f1_score(fl, fp, average="macro")
        print(f"  FT Epoch {ep}/50: Acc={ft_acc*100:.2f}% F1={ft_f1:.4f}")

print(f"\nSimCLR + Fine-tune: {ft_best*100:.2f}%")
print(f"Supervised only:    {best_acc*100:.2f}%")
print(f"Delta:              {(ft_best - best_acc)*100:+.2f}%")

## 12. LOSO Evaluation

In [ ]:
subjects = UCIHARDataset.get_subjects(DATA_DIR)
print(f"Subjects: {subjects}")
print(f"Running LOSO ({len(subjects)}-fold)...\n")

loso_results = []
for subj in subjects:
    loso_train, loso_test = UCIHARDataset.loso_split(DATA_DIR, subj)
    lm, ls = loso_train.X.mean(dim=(0, 1)), loso_train.X.std(dim=(0, 1))
    ls[ls < 1e-8] = 1.0
    loso_train.X = (loso_train.X - lm) / ls
    loso_test.X = (loso_test.X - lm) / ls

    loso_aug = AugmentedDataset(loso_train, SensorAugmentor(p=0.5))
    l_train_loader = DataLoader(loso_aug, batch_size=64, shuffle=True)
    l_test_loader = DataLoader(loso_test, batch_size=64, shuffle=False)

    lm_model = SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=6).to(device)
    lm_opt = torch.optim.AdamW(lm_model.parameters(), lr=0.001, weight_decay=1e-4)
    lm_sched = torch.optim.lr_scheduler.CosineAnnealingLR(lm_opt, T_max=50)
    lm_best = 0.0

    for ep in range(1, 51):
        lm_model.train()
        for X, y in l_train_loader:
            X, y = X.to(device), y.to(device)
            lm_opt.zero_grad()
            nn.CrossEntropyLoss()(lm_model(X), y).backward()
            lm_opt.step()
        lm_sched.step()
        lm_model.eval()
        lp, ll = [], []
        with torch.no_grad():
            for X, y in l_test_loader:
                X, y = X.to(device), y.to(device)
                lp.append(lm_model(X).argmax(1).cpu().numpy())
                ll.append(y.cpu().numpy())
        lp, ll = np.concatenate(lp), np.concatenate(ll)
        la = (lp == ll).mean()
        if la > lm_best:
            lm_best = la

    lf = f1_score(ll, lp, average="macro")
    loso_results.append((subj, lm_best, lf, len(loso_test)))
    print(f"  Subject {subj:2d}: Acc={lm_best*100:.2f}% F1={lf:.4f} (n={len(loso_test)})")

loso_accs = [r[1] for r in loso_results]
loso_f1s = [r[2] for r in loso_results]
print(f"\nLOSO Mean Accuracy: {np.mean(loso_accs)*100:.2f}% +/- {np.std(loso_accs)*100:.2f}%")
print(f"LOSO Mean F1:       {np.mean(loso_f1s):.4f} +/- {np.std(loso_f1s):.4f}")

## 13. Cross-Dataset Transfer

In [ ]:
print("Cross-Dataset Transfer: UCI-HAR -> PAMAP2\n")

PAMAP2_DIR = "data/PAMAP2_Dataset"
if not os.path.isdir(PAMAP2_DIR):
    PAMAP2Dataset.download("data")

pamap2_test = PAMAP2Dataset(PAMAP2_DIR, split="test")
pm, ps = PAMAP2Dataset.get_normalization_stats(PAMAP2_DIR)
pamap2_test.X = (pamap2_test.X - torch.tensor(pm, dtype=torch.float32)) / torch.tensor(ps, dtype=torch.float32)

uci_to_pamap = {0: 3, 1: 7, 2: 8, 3: 1, 4: 2, 5: 0}
pamap_to_uci = {v: k for k, v in uci_to_pamap.items()}

transfer_mask = torch.tensor([y.item() in pamap_to_uci for y in pamap2_test.y])
if transfer_mask.any():
    t_X = pamap2_test.X[transfer_mask]
    t_y = torch.tensor([pamap_to_uci[y.item()] for y in pamap2_test.y[transfer_mask]])

    model.eval()
    with torch.no_grad():
        t_out = model(t_X.to(device))
        t_preds = t_out.argmax(1).cpu().numpy()

    t_acc = (t_preds == t_y.numpy()).mean()
    t_f1 = f1_score(t_y.numpy(), t_preds, average="macro")
    print(f"Overlapping activities: {list(pamap_to_uci.keys())}")
    print(f"Mapped to UCI-HAR:      {list(pamap_to_uci.values())}")
    print(f"Samples evaluated:      {len(t_y)}")
    print(f"Transfer Accuracy:      {t_acc*100:.2f}%")
    print(f"Transfer F1 Macro:      {t_f1:.4f}")
else:
    print("No overlapping activities found")

## 14. Baseline Comparison

In [ ]:
baselines = [
    ("DeepConvLSTM", "~1.5M", "~5.8 MB", "92.1%", "~50ms"),
    ("InnoHAR", "~200K", "~780 KB", "93.1%", "~30ms"),
    ("Attend-Discrim.", "~500K", "~1.9 MB", "93.8%", "~45ms"),
    ("TinyHAR", "~50K", "~195 KB", "93.5%", "~15ms"),
    ("MobileHAR", "~100K", "~390 KB", "91.8%", "~12ms"),
    ("CNN-GRU", "~300K", "~1.2 MB", "92.5%", "~35ms"),
    ("Ours", f"{model.count_parameters():,}", f"{model.quantized_size_kb():.0f} KB (INT8)", f"{best_acc*100:.1f}%", "<10ms"),
]

print(f"{'Model':<20s}{'Params':<12s}{'Size':<20s}{'Accuracy':<12s}{'Latency':<10s}")
print("-" * 74)
for row in baselines:
    print(f"{row[0]:<20s}{row[1]:<12s}{row[2]:<20s}{row[3]:<12s}{row[4]:<10s}")

fig, ax = plt.subplots(figsize=(10, 6))
sizes_kb = [5800, 780, 1900, 195, 390, 1200, model.quantized_size_kb()]
accs_plot = [92.1, 93.1, 93.8, 93.5, 91.8, 92.5, best_acc * 100]
names_plot = ["DeepConvLSTM", "InnoHAR", "Attend-D.", "TinyHAR", "MobileHAR", "CNN-GRU", "Ours"]
colors_plot = ["#888888"] * 6 + ["#FF6B6B"]
sizes_scatter = [80] * 6 + [200]

for i in range(len(names_plot)):
    ax.scatter(sizes_kb[i], accs_plot[i], c=colors_plot[i], s=sizes_scatter[i], zorder=5)
    ax.annotate(names_plot[i], (sizes_kb[i], accs_plot[i]), textcoords="offset points", xytext=(8, 5), fontsize=9)

ax.set_xscale("log")
ax.set_xlabel("Model Size (KB, log scale)")
ax.set_ylabel("Accuracy (%)")
ax.set_title("Model Size vs Accuracy (Pareto Front)")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig("pareto_front.png", dpi=150, bbox_inches="tight")
plt.show()

## 15. ONNX Export

In [ ]:
dummy = torch.randn(1, 128, 6).to(device)
model.eval()
onnx_path = "checkpoints/sensorfusion_har.onnx"
torch.onnx.export(model, dummy, onnx_path, input_names=["sensor_input"], output_names=["activity_logits"], dynamic_axes={"sensor_input": {0: "batch"}, "activity_logits": {0: "batch"}}, opset_version=14)
onnx_size = os.path.getsize(onnx_path) / 1024
print(f"ONNX exported: {onnx_path}")
print(f"ONNX size: {onnx_size:.2f} KB")

try:
    import onnxruntime as ort
    sess = ort.InferenceSession(onnx_path)
    onnx_out = sess.run(None, {"sensor_input": dummy.cpu().numpy()})
    torch_out = model(dummy).detach().cpu().numpy()
    match = np.allclose(torch_out, onnx_out[0], atol=1e-5)
    print(f"Verification: {'PASSED' if match else 'FAILED'}")
except ImportError:
    print("onnxruntime not installed, skipping verification")

## 16. Inference Benchmark

In [ ]:
model.eval()
sample = test_ds.X[0:1].to(device)
for _ in range(50):
    with torch.no_grad():
        model(sample)
latencies = []
for _ in range(1000):
    t0 = time.perf_counter()
    with torch.no_grad():
        model(sample)
    latencies.append((time.perf_counter() - t0) * 1000)
latencies = np.array(latencies)
print(f"Inference Latency (1000 runs):")
print(f"  Mean:  {latencies.mean():.3f} ms")
print(f"  Std:   {latencies.std():.3f} ms")
print(f"  P95:   {np.percentile(latencies, 95):.3f} ms")
print(f"  P99:   {np.percentile(latencies, 99):.3f} ms")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(latencies, bins=50, color="#4ECDC4", edgecolor="none", alpha=0.8)
ax.axvline(latencies.mean(), color="#FF6B6B", linestyle="--", linewidth=2, label=f"Mean: {latencies.mean():.2f}ms")
ax.axvline(np.percentile(latencies, 95), color="#FFE66D", linestyle="--", linewidth=2, label=f"P95: {np.percentile(latencies, 95):.2f}ms")
ax.set_xlabel("Latency (ms)")
ax.set_ylabel("Count")
ax.set_title("Inference Latency Distribution")
ax.legend()
plt.tight_layout()
plt.savefig("latency_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 17. Final Summary

In [ ]:
print("=" * 60)
print("  SensorFusion-HAR -- Final Results")
print("=" * 60)
print(f"  Dataset:               UCI-HAR (6 classes)")
print(f"  Supervised Accuracy:   {best_acc*100:.2f}%")
print(f"  SimCLR + FT Accuracy:  {ft_best*100:.2f}%")
print(f"  LOSO Mean Accuracy:    {np.mean(loso_accs)*100:.2f}% +/- {np.std(loso_accs)*100:.2f}%")
print(f"  Parameters:            {model.count_parameters():,}")
print(f"  Model Size (FP32):     {model.model_size_kb():.2f} KB")
print(f"  Model Size (INT8):     {model.quantized_size_kb():.2f} KB")
print(f"  ONNX Size:             {onnx_size:.2f} KB")
print(f"  Inference Latency:     {latencies.mean():.2f} ms (mean)")
print(f"  Calibration (ECE):     {ece:.4f}")
print("=" * 60)